# Milestone 2 lab, notebook 7 of 8: The false alarm

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| **7. The false alarm** (you are here) | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
import pandas as pd
from lab_helpers import keep, lab_copy, plant, prepare_week, production, sql

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Nine alarms

Notebook 6 ended with two new checks and nine alarms: seven store-days far from a normal day, and two store-days with no rows.

Imagine it is 02:10 at night and all nine reach your phone. Start with the seven. Here they are again.

In [2]:
NORMAL_DAY = """
    SELECT today.store_id, today.sales_date, today.lines,
           count(before.lines) AS days_compared,
           round(avg(before.lines)) AS normal_lines,
           round(100.0 * today.lines / avg(before.lines)) AS percent_of_normal
    FROM (
        SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines
        WHERE sales_date <= DATE '2026-08-30'
        GROUP BY store_id, sales_date
    ) today
    JOIN (
        SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines
        GROUP BY store_id, sales_date
    ) before
      ON before.store_id = today.store_id
     AND before.sales_date IN (today.sales_date - 7, today.sales_date - 14,
                               today.sales_date - 21, today.sales_date - 28)
    {where}
    GROUP BY today.store_id, today.sales_date, today.lines
"""
OUTSIDE = "days_compared >= 3 AND (percent_of_normal < {low} OR percent_of_normal > {high})"

no_calendar = NORMAL_DAY.format(where="")
sql(f"SELECT store_id, sales_date, lines, normal_lines, percent_of_normal FROM ({no_calendar}) n "
    f"WHERE {OUTSIDE.format(low=60, high=140)} ORDER BY sales_date, store_id")

,store_id,sales_date,lines,normal_lines,percent_of_normal
0,S01,2026-08-26,52,139,37
1,S02,2026-08-26,36,125,29
2,S03,2026-08-26,42,110,38
3,S04,2026-08-26,41,106,39
4,S05,2026-08-26,30,78,38
5,S06,2026-08-26,32,92,35
6,S02,2026-08-28,334,148,226


Six of the seven are Wednesday 26 August, and **every** store is on the list, each between 29% and 39% of normal.

That pattern is a clue. One store far below normal is that store's problem. All six stores far below normal on the same day is not six problems. It is the **day**.

Head office keeps a small calendar of days that are not normal.

In [3]:
sql("SELECT calendar_date, store_id, kind, opens_at, closes_at, note FROM clean.calendar ORDER BY 1, 2")

,calendar_date,store_id,kind,opens_at,closes_at,note
0,2026-08-26,S01,public_holiday,10:00:00,16:00:00,Founders' Day
1,2026-08-26,S02,public_holiday,10:00:00,16:00:00,Founders' Day
2,2026-08-26,S03,public_holiday,10:00:00,16:00:00,Founders' Day
3,2026-08-26,S04,public_holiday,10:00:00,16:00:00,Founders' Day
4,2026-08-26,S05,public_holiday,10:00:00,16:00:00,Founders' Day
5,2026-08-26,S06,public_holiday,10:00:00,16:00:00,Founders' Day
6,2026-08-27,S01,closed,None,None,Planned refit


Wednesday 26 August was **Founders' Day**, a public holiday. The stores opened from 10:00 to 16:00, and sold about a third of a normal Wednesday. Nothing is wrong with those six files.

And the last row of the calendar explains one of the two "no rows" alarms: store `S01` was **closed** on Thursday 27 August for a planned refit. It sent no file, and that is correct.

An alarm when nothing is wrong is called a **false positive**. Most people just say **false alarm**.

Count them. Nine alarms. Two are real: Dana's doubled Friday, and Ines's missing Tuesday. **Seven are false.**

That is worse than it sounds. The person who is woken seven times for nothing does not read the eighth alarm. A check that cries wolf gets switched off, and then it catches nothing at all.

## Step 2: Teach the checks about the calendar

The fix is not a wider limit. It is to tell the checks what the business already knows.

The volume check leaves out every day that is in the calendar: a holiday is not judged as a normal day, and it is not used to work out a normal day either.

In [4]:
with_calendar = NORMAL_DAY.format(where="""
    WHERE NOT EXISTS (SELECT 1 FROM clean.calendar c WHERE c.store_id = today.store_id
                      AND c.calendar_date IN (today.sales_date, before.sales_date))""")
yours = sql(f"SELECT store_id, sales_date, lines, normal_lines, percent_of_normal FROM ({with_calendar}) n "
            f"WHERE {OUTSIDE.format(low=60, high=140)}")
yours

,store_id,sales_date,lines,normal_lines,percent_of_normal
0,S02,2026-08-28,334,148,226


In [5]:
keep("normal_volume", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (A store's day is near its normal day)
The same rows.


One alarm, and it is the real one. Now "every open store reported": a store is not expected on a day the calendar says it is closed.

In [6]:
yours = sql("""
    SELECT s.store_id, d.day AS sales_date, 'no sales lines for an open store' AS problem
    FROM clean.stores s
    CROSS JOIN (SELECT DISTINCT business_date AS day FROM clean.till_totals) d
    LEFT JOIN (SELECT DISTINCT store_id, sales_date FROM clean.sales_lines) l
      ON l.store_id = s.store_id AND l.sales_date = d.day
    WHERE l.store_id IS NULL
      AND NOT EXISTS (SELECT 1 FROM clean.calendar c WHERE c.store_id = s.store_id
                      AND c.calendar_date = d.day AND c.kind = 'closed')
""")
yours

,store_id,sales_date,problem
0,S04,2026-08-25,no sales lines for an open store


In [7]:
keep("every_store_reported", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (Every open store reported)
The same rows.


Nine alarms became two, and both are real.

## Step 3: What silence costs

You made the volume check quiet on holidays. Quiet is not free. Ask: **what can it no longer see?**

Try it. In the private copy, store `S03` loses most of its file on the holiday: only the first line of every receipt arrives.

In [8]:
lab_copy()
plant("""
    DELETE FROM lab.sales_lines
    WHERE store_id = 'S03' AND sales_date = DATE '2026-08-26' AND line_no > 1
""")
left = sql("SELECT count(*) AS lines_left FROM lab.sales_lines WHERE store_id = 'S03' AND sales_date = DATE '2026-08-26'")
print("S03 had 42 lines on the holiday. In the copy it has", int(left.iloc[0, 0]), "left.")
production("normal_volume", on_the_lab=True)[["store_id", "sales_date", "lines", "percent_of_normal"]]

lab.sales_lines is a fresh copy of clean.sales_lines: 24,384 rows.
Planted in lab.sales_lines.
S03 had 42 lines on the holiday. In the copy it has 15 left.


,store_id,sales_date,lines,percent_of_normal
0,S02,2026-08-28,334,226


Still one row, and it is Dana's Friday. Two thirds of a store's holiday sales are gone, and the check says nothing. You told it not to look at that day.

A fault with no alarm is called a **false negative**.

The two mistakes pull against each other:

| | The check is silent | The check fires |
|---|---|---|
| **Nothing is wrong** | Good | False positive: a false alarm |
| **Something is wrong** | False negative: a missed fault | Good |

Every time you make a check quieter, you buy fewer false alarms with more missed faults. Do it with open eyes, and write down what you gave up.

Here the price is small, because another check still watches the holiday. If the file had not arrived **at all**:

In [9]:
plant("DELETE FROM lab.sales_lines WHERE store_id = 'S03' AND sales_date = DATE '2026-08-26'")
production("every_store_reported", on_the_lab=True)

Planted in lab.sales_lines.


,store_id,sales_date,problem
0,S03,2026-08-26,no sales lines for an open store
1,S04,2026-08-25,no sales lines for an open store


"Every open store reported" still catches it: the calendar says the holiday was a short day, not a closed one.

## Step 4: Move the limit, and count

The limit is the other place where you choose between the two mistakes. Try six limits on the days that are not in the calendar. The only real fault among those days is Dana's doubled Friday.

In [10]:
rows = []
for low, high in [(90, 110), (80, 120), (70, 130), (60, 140), (50, 200), (30, 250)]:
    found = sql(f"SELECT store_id, sales_date FROM ({with_calendar}) n WHERE {OUTSIDE.format(low=low, high=high)}")
    real = int(((found["store_id"] == "S02") & (found["sales_date"].astype(str) == "2026-08-28")).sum())
    rows.append({"limit": f"{low}% to {high}%", "alarms": len(found),
                 "the doubled Friday": "caught" if real else "MISSED", "false alarms": len(found) - real})
pd.DataFrame(rows)

,limit,alarms,the doubled Friday,false alarms
0,90% to 110%,31,caught,30
1,80% to 120%,6,caught,5
2,70% to 130%,2,caught,1
3,60% to 140%,1,caught,0
4,50% to 200%,1,caught,0
5,30% to 250%,0,MISSED,0


Read it from top to bottom.

- **Too tight** (90% to 110%): 31 alarms in two weeks, and 30 of them are false. Nobody will read that.
- **In the middle** (60% to 140%): one alarm, the real one.
- **Too loose** (30% to 250%): no alarms. Quiet, and it misses a file that arrived twice.

No limit is free of both mistakes for ever. You choose one, from data, and you write down why.

## Step 5: Why the same weekday?

One choice in notebook 6 went by quickly: a day is compared with the **same weekday** in earlier weeks. Why not simply with the average of the last four weeks?

Look at how busy each weekday is, in the clean weeks.

In [11]:
sql("""
    SELECT to_char(min(sales_date), 'Dy') AS weekday, round(avg(lines)) AS average_lines_per_store
    FROM (
        SELECT sales_date, store_id, count(*) AS lines
        FROM clean.sales_lines
        WHERE sales_date < DATE '2026-08-24'
        GROUP BY sales_date, store_id
    ) d
    GROUP BY extract(isodow FROM sales_date)
    ORDER BY extract(isodow FROM sales_date)
""")

,weekday,average_lines_per_store
0,Mon,104
1,Tue,101
2,Wed,108
3,Thu,111
4,Fri,132
5,Sat,152
6,Sun,120


A Saturday has about half as many lines again as a Tuesday. A pattern that repeats with the calendar is called **seasonality**. Saturdays are busy, holidays are quiet, December is not like February.

See what a baseline that ignores it does. Compare every day of the bonus week with the plain average of the 28 days before.

In [12]:
sql("""
    SELECT today.store_id, today.sales_date, to_char(today.sales_date, 'Dy') AS weekday, today.lines,
           round(avg(before.lines)) AS flat_normal,
           round(100.0 * today.lines / avg(before.lines)) AS percent_of_normal
    FROM (SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines GROUP BY 1, 2) today
    JOIN (SELECT store_id, sales_date, count(*) AS lines FROM clean.sales_lines GROUP BY 1, 2) before
      ON before.store_id = today.store_id
     AND before.sales_date BETWEEN today.sales_date - 28 AND today.sales_date - 1
    WHERE today.sales_date BETWEEN DATE '2026-08-24' AND DATE '2026-08-30'
      AND today.sales_date <> DATE '2026-08-26'
    GROUP BY 1, 2, 3, 4
    HAVING 100.0 * today.lines / avg(before.lines) NOT BETWEEN 60 AND 140
    ORDER BY today.sales_date
""")

,store_id,sales_date,weekday,lines,flat_normal,percent_of_normal
0,S02,2026-08-28,Fri,334,125,267
1,S04,2026-08-29,Sat,172,116,149


Two alarms. One is Dana's Friday. The other is store `S04` on Saturday 29 August, at 149% of a "normal day".

That Saturday is an ordinary Saturday. Against other Saturdays it was at 106%. The flat baseline called it strange only because Saturdays are always busy. A false alarm, made by the baseline and not by the data.

## What you now know

- A false positive is an alarm when nothing is wrong. A false negative is a fault with no alarm.
- Too many false alarms kill a check: people stop reading it.
- Fix a false alarm with knowledge (a calendar) before you fix it with a wider limit.
- Every exclusion and every wider limit costs missed faults. Say which ones.
- Seasonality is a pattern that repeats with the calendar. Compare like with like.

**Say it in an interview:** "My first volume check fired nine times in a week, and seven were false: a public holiday in all six stores, and one store closed for a refit. I did not loosen the limit. I joined a small calendar from head office, and it went down to the two real faults. I also wrote down the cost: on a holiday the volume check is blind, so a different check still confirms that every store sent a file."

**The theory:** [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), sections "What is normal?" and "Alerts people answer".

Next: [notebook 8](08_the_production_version.ipynb). All sixteen checks, as one tested program. And what they still do not change.